# ISC 7700X — HW3 Review Notebook
### Topics: Linear Models, Confusion Matrix, Economic Cost Optimization, Threshold Tuning
---
This notebook walks through every concept covered in HW3. Use it as a reference all semester.
Each section has: **the concept**, **the code**, **the result**, and **the key insight**.

## 1. The Big Picture

**What are we doing?**
We have labeled data (features + true labels) and a hand-crafted linear model. We want to:
1. Apply the model to get predictions
2. Measure how accurate it is
3. Measure its **economic** performance (accuracy alone is not enough)
4. Optimize the model for economic gain — not just accuracy

**Key insight:** A model with 88% accuracy can still be economically terrible if it makes the *wrong kind* of errors.

## 2. Loading & Exploring the Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv('hw3.data1.csv.gz')

print('Shape:', df.shape)          # rows x columns
print('Columns:', df.columns.tolist())
print('Label distribution:')
print(df['label'].value_counts())

**What to look for:**
- 21 columns: 20 features (column1–column20) + 1 label
- Label values: only `1` and `-1`
- Class balance: are there equal numbers of each class?

**Key concept — Majority Class Baseline:**
If you predicted the majority class for every row, what accuracy would you get?
This is your sanity check. Any real model should beat this.

## 3. The Linear Model

**Concept: What is a linear model?**

A linear model computes a weighted sum of input features plus a bias term:

```
y = w1*x1 + w2*x2 + ... + wn*xn + bias
```

This is called a **dot product** of weights and features, plus bias.

Then a **threshold** converts the continuous value into a class label:
- if y > threshold → predict 1
- else → predict -1

**The model given in HW3:**
```
y = 24*col1 + -15*col2 + -38*col3 + -7*col4 + -41*col5 + 35*col6 
  + 0*col7 + -2*col8 + 19*col9 + 33*col10 + -3*col11 + 7*col12 
  + 3*col13 + -47*col14 + 26*col15 + 10*col16 + 40*col17 
  + -1*col18 + 3*col19 + 0*col20 + (-6)
```
Threshold: y > 0 → 1, else -1

In [ ]:
# Define weights and bias
weights = [24, -15, -38, -7, -41, 35, 0, -2, 19, 33, -3, 7, 3, -47, 26, 10, 40, -1, 3, 0]
bias = -6

# Select feature columns
feature_cols = [f'column{i}' for i in range(1, 21)]

# Compute dot product + bias (this gives a continuous score for each row)
y_raw = df[feature_cols].dot(weights) + bias

# Apply threshold: if y > 0 → 1, else -1
df['prediction'] = y_raw.apply(lambda x: 1 if x > 0 else -1)

print('Prediction distribution:')
print(df['prediction'].value_counts())

# Note: 0 > 0 is False, so y=0 gets classified as -1
print('\n0 > 0 evaluates to:', 0 > 0)

**Key insight:** The dot product gives a *score* — how confidently the model leans toward one class.
The threshold converts that score into a binary decision. These are **two separate concerns**.

## 4. Accuracy

**Concept:** Accuracy = fraction of correct predictions

```
Accuracy = (correct predictions) / (total predictions)
```

In pandas, comparing two columns element-wise gives a boolean series.
Taking `.mean()` of a boolean series gives the fraction of `True` values.

In [ ]:
df['correct'] = df['prediction'] == df['label']
accuracy = df['correct'].mean()
print(f'Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)')

# Result: 88% accuracy

**Warning:** Accuracy alone is misleading on imbalanced datasets or when errors have different costs.
88% sounds great — but is it economically good? We need the confusion matrix to find out.

## 5. Confusion Matrix

**Concept:** Breaks down predictions into four categories:

```
                  Predicted
                  1        -1
Actual   1       TP        FN
        -1       FP        TN
```

- **TP (True Positive):** Actually 1, predicted 1 ✓
- **TN (True Negative):** Actually -1, predicted -1 ✓  
- **FP (False Positive):** Actually -1, predicted 1 ✗
- **FN (False Negative):** Actually 1, predicted -1 ✗

**Sanity checks:**
- TP + FN = total actual positives
- FP + TN = total actual negatives
- TP + TN + FP + FN = total rows

In [ ]:
TP = ((df['label'] == 1)  & (df['prediction'] == 1)).sum()
TN = ((df['label'] == -1) & (df['prediction'] == -1)).sum()
FP = ((df['label'] == -1) & (df['prediction'] == 1)).sum()
FN = ((df['label'] == 1)  & (df['prediction'] == -1)).sum()

print(f'TP: {TP}  TN: {TN}  FP: {FP}  FN: {FN}')
print(f'Total: {TP+TN+FP+FN} (should equal {len(df)})')

# Derived metrics
precision = TP / (TP + FP)
recall    = TP / (TP + FN)
f1        = 2 * precision * recall / (precision + recall)

print(f'\nPrecision: {precision:.4f}')  # of predicted positives, how many were right?
print(f'Recall:    {recall:.4f}')       # of actual positives, how many did we catch?
print(f'F1 Score:  {f1:.4f}')           # harmonic mean of precision and recall

# Results: TP=3550, TN=5259, FP=484, FN=707

## 6. Economic Cost Analysis

**Concept:** In the real world, different errors have different costs.
- Cost of False Negative (FN): **$1,000**
- Cost of False Positive (FP): **$100**
- Cost of correct prediction: **$0**

**Economic loss = (FN × $1,000) + (FP × $100)**

This is why accuracy alone is not enough — a model optimized for accuracy
may not be optimized for the business objective.

In [ ]:
fn_cost = FN * 1000
fp_cost = FP * 100
economic_loss = fn_cost + fp_cost

print(f'FN cost: ${fn_cost:,}')
print(f'FP cost: ${fp_cost:,}')
print(f'Total economic loss: ${economic_loss:,}')
print(f'\nFN contributes: {fn_cost/economic_loss*100:.2f}% of loss')
print(f'FP contributes: {fp_cost/economic_loss*100:.2f}% of loss')

# Result: $755,400 total loss
# 93.59% from FN — FN is the dominant problem

**Key insight:** 93.59% of economic loss comes from false negatives, even though
there are fewer FN (707) than FP (484) would suggest proportionally.
The $1,000 cost per FN dominates everything.

**Action:** We need to reduce FN, even at the cost of more FP.
Each FN converted to FP saves us $900 net ($1,000 - $100).

## 7. Threshold Optimization

**Concept:** The threshold is a separate decision from the model weights.
By lowering the threshold, we predict `1` more aggressively:
- Fewer FN (we catch more actual positives)
- More FP (we over-predict positives)

Given FN costs 10x more than FP, lowering the threshold should reduce total cost.

**This is called threshold sweeping — a systematic search for the optimal decision boundary.**

In [ ]:
results = []

for t in range(-2000, 500, 10):
    predictions = y_raw.apply(lambda x: 1 if x > t else -1)
    FN_t = ((df['label'] == 1)  & (predictions == -1)).sum()
    FP_t = ((df['label'] == -1) & (predictions == 1)).sum()
    loss = (FN_t * 1000) + (FP_t * 100)
    results.append({'threshold': t, 'loss': loss, 'FN': FN_t, 'FP': FP_t})

results_df = pd.DataFrame(results)
best = results_df.loc[results_df['loss'].idxmin()]
print('Optimal threshold result:')
print(best)
print(f'\nOriginal loss:  $755,400')
print(f'Optimized loss: ${int(best["loss"]):,}')
print(f'Economic gain:  ${755400 - int(best["loss"]):,}')

In [ ]:
# Visualize how loss changes with threshold
plt.figure(figsize=(10, 5))
plt.plot(results_df['threshold'], results_df['loss'], color='steelblue', linewidth=1.5)
plt.axvline(x=best['threshold'], color='red', linestyle='--', label=f'Optimal: {int(best["threshold"])}')
plt.axvline(x=0, color='gray', linestyle='--', label='Original threshold: 0')
plt.xlabel('Threshold')
plt.ylabel('Economic Loss ($)')
plt.title('Economic Loss vs Decision Threshold')
plt.legend()
plt.tight_layout()
plt.show()

**Results:**
- Optimal threshold: **-1010**
- At threshold -1010: FN = 0, FP = 1124
- Optimized loss: **$112,400**
- Original loss: **$755,400**
- **Economic gain: $643,000**

**Why does it stop at FN=0?**
Once FN reaches 0, lowering the threshold further only adds more FP without reducing FN.
FN cannot go below 0, so the loss starts rising again.

## 8. Key Concepts Summary

| Concept | Definition | Why it matters |
|---|---|---|
| Linear model | Weighted sum of features + bias | Simple, interpretable baseline |
| Dot product | w·x = sum of weight × feature | Efficient computation of linear model |
| Threshold | Decision boundary on raw score | Separates positive from negative predictions |
| Accuracy | Fraction of correct predictions | Easy to understand but incomplete |
| Confusion matrix | TP/TN/FP/FN breakdown | Reveals *type* of errors |
| Precision | TP/(TP+FP) | Of predicted positives, how many were right? |
| Recall | TP/(TP+FN) | Of actual positives, how many did we catch? |
| Economic loss | (FN×cost_fn) + (FP×cost_fp) | Business-relevant performance metric |
| Threshold sweep | Try many thresholds, pick best | Optimizes for economic objective |

## 9. Lessons to Carry Forward

1. **Accuracy is not the goal** — the business objective is. Always ask: what does an error cost?
2. **Model weights and decision threshold are separate** — you can improve performance without retraining
3. **Always check which error dominates your cost** — it tells you exactly which direction to optimize
4. **Sanity check your confusion matrix** — TP+FN+FP+TN must equal total rows
5. **Threshold sweeping is powerful** — a simple loop can dramatically improve economic performance
6. **FN=0 is not always the goal** — in this case it was optimal, but in other cost structures it might not be